# FACS Depletion Curves vs HD DIT-HAP

Overlay per-gene FACS depletion curves (2 strain replicates) with the corresponding HD DIT-HAP gene-level LFC curve. One subplot per gene.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

# Resolve repo root via git so all relative paths work regardless of kernel CWD
_repo_root = Path(subprocess.check_output(["git", "rev-parse", "--show-toplevel"], text=True).strip())
os.chdir(_repo_root)
sys.path.insert(0, str(_repo_root))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from workflow.src.data_config import load_dataset_config

In [ ]:
# HD DIT-HAP generation values (from projects/HD_DIT_HAP/config/config.yaml time_points)
HD_GENERATIONS = [0.0, 2.352, 5.588, 9.104, 12.480]
HD_LFC_COLS = ["YES0", "YES1", "YES2", "YES3", "YES4"]

FACS_PATH = "resources/curated/FACS_depletion_curves_from_XH.xlsx"
MPLSTYLE = "config/DIT_HAP.mplstyle"

In [ ]:
# Load data
facs_df = pd.read_excel(FACS_PATH)

cfg = load_dataset_config("HD_DIT_HAP")
hd_lfc_df = pd.read_csv(cfg.gene_level.LFCs, sep="\t")

print("FACS genes:", sorted(facs_df["Name"].unique().tolist()))
print("HD DIT-HAP shape:", hd_lfc_df.shape)

In [ ]:
plt.style.use(MPLSTYLE)

genes = sorted(facs_df["Name"].unique().tolist())
n_cols = 5
n_rows = int(np.ceil(len(genes) / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(4 * n_cols, 3.5 * n_rows), sharey=False)
axes = axes.flatten()

FACS_COLORS = ["#e07b54", "#5b8fd4"]
HD_COLOR = "black"

for idx, gene in enumerate(genes):
    ax = axes[idx]

    # --- FACS curves ---
    gene_facs = facs_df[facs_df["Name"] == gene]
    for color, (alias, strain_df) in zip(FACS_COLORS, gene_facs.groupby("Strain_alias", sort=True)):
        ax.plot(
            strain_df["Generations"],
            strain_df["LFC"],
            color=color,
            linestyle="--",
            marker="o",
            markersize=4,
            linewidth=1.5,
            label=f"FACS {alias}",
        )

    # --- HD DIT-HAP curve ---
    gene_hd = hd_lfc_df[hd_lfc_df["Name"] == gene]
    if not gene_hd.empty:
        hd_lfc_values = gene_hd[HD_LFC_COLS].values.flatten().tolist()
        ax.plot(
            HD_GENERATIONS,
            hd_lfc_values,
            color=HD_COLOR,
            linestyle="-",
            marker="s",
            markersize=4,
            linewidth=1.5,
            label="HD DIT-HAP",
        )

    ax.axhline(0, color="gray", linewidth=0.6, linestyle=":")
    ax.set_title(gene, fontsize=11, fontweight="bold")
    ax.set_xlabel("Generations")
    ax.set_ylabel("LFC")
    ax.legend(fontsize=7, frameon=False)

for j in range(idx + 1, len(axes)):
    fig.delaxes(axes[j])

fig.suptitle("FACS depletion curves vs HD DIT-HAP", fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig("notebooks/validation/facs_vs_dit_hap.pdf", bbox_inches="tight")
plt.show()